# Setup for using GN for GINNs

### Imports and setup

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch import optim
from tqdm.notebook import trange
import k3d
import sys
import os
import time
import copy
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from models.model_architecture import PlaceholderNet # make sure the residuals are defined as model attributes
from training.optimizers import GaussNewtonWoodburyBig # will be renamed at some point
from util.surface_sampling import sample_model_surface_binsearch, sample_model_surface_newton 
from util.visualization.utils_mesh import get_mesh

torch.manual_seed(0)


device = 'cuda' #if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

### Main training loop

In [5]:
pts_eikonal = torch.zeros(1000, 3)
pts_boundary = torch.zeros(1000, 3)
pts_surface = torch.zeros(1000, 3)
# include connectedness and design region in data. 
# If their weighting should be different this needs to be adapted inside the optimizer code
pts_data = torch.zeros(1000, 3)
vals_data = torch.zeros(pts_data.shape[0])
data = {"pts_data": pts_data, "vals_data": vals_data}

loss_weights = {"interface": 1.0, "data": 1.0, "eikonal": 0.001, "surface_strain": 1.0}

config = {
    "pts_boundary": pts_boundary,
    "pts_eikonal": pts_eikonal,
    "pts_surface": pts_surface,
    "data": data,
    "loss_weights": loss_weights,
    "regularization": 1e-6,
}

model = PlaceholderNet(ks=[3, 128, 128, 128, 128, 1])
model = model.double()
params = model.params

In [6]:
# I have added line search to the optimizer for better stability
optimizer = GaussNewtonWoodburyBig(model, lr=1e-1, config=config)

for i in (pbar:=trange(1000)):
    optimizer.zero_grad()
    
    with torch.no_grad():
        # These loss calculations are just for logging, a more efficient way would be to calculate them from the residual
        # inside the optimizer class and output from there
        loss_interface  = 0.5*model.f(params, pts_boundary).square().mean()
        loss_data  = 0.5*model.r_data(params, pts_data, vals_data).square().mean()
        loss_eikonal  = 0.5*model.r_eikonal(params, pts_eikonal).squeeze(1).square().mean()

        # Sample new surface points, you can replace this with the surface sampling from GINN
        pts_surface = sample_model_surface_binsearch(model, pts_boundary, bound_limit=2)
        # Optional: refine surface samples with Newton
        pts_surface = sample_model_surface_newton(model, pts_surface)
        # Update the config
        config["pts_surface"] = pts_surface
        optim.config = config
        
        loss_surface_strain = 0.5*(model.r_principle_curvature_1(params, pts_surface).squeeze(1).square().mean()
                                   + model.r_principle_curvature_2(params, pts_surface).squeeze(1).square().mean())
           
        loss = loss_weights["interface"] * loss_interface + loss_weights["eikonal"] * loss_eikonal + loss_weights["surface_strain"] * loss_surface_strain

        pbar.set_description(f"interface: {loss_interface.item():.2e} "
                            f"data: {loss_data.item():.2e} "
                            f"eikonal: {loss_eikonal.item():.2e} "
                            f"surface strain: {loss_surface_strain.item():.2e} "
                            f"{len(pts_surface)}"
                            )
    optimizer.step()

plt.plot(loss_over_iters.keys(), loss_over_iters.values())
plt.semilogy()
plt.show()

  0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 